# Lecture 16: Word Vectors — Hands-On Demo

This notebook accompanies the slides on word vectors, Word2Vec, and BERT.
We'll load pre-trained word vectors, explore word similarity and analogies,
and visualize word relationships using PCA.

**Packages**: `gensim`, `matplotlib`, `scikit-learn`

If you haven't installed `gensim` yet:
```
conda install gensim
```

In [ ]:
import gensim.downloader as api
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## Part 1: Loading Pre-Trained Word Vectors

We'll use **GloVe** vectors trained on Wikipedia and Gigaword (6 billion
words of text). These are 50-dimensional vectors — each word is
represented as a list of 50 numbers.

Larger models exist (100d, 200d, 300d), but the 50d version is small
enough to download and load quickly in class (~66 MB).

**The first time you run this, it will download the model. This takes
about a minute.**

In [ ]:
# Download and load pre-trained GloVe vectors
# This takes ~1 minute on the first run (downloading ~66 MB)
model = api.load("glove-wiki-gigaword-50")

print(f"Vocabulary size: {len(model):,} words")
print(f"Vector dimensions: {model.vector_size}")

### What does a word vector look like?

Each word is represented as a list of numbers. These numbers don't have
obvious individual meanings — it's the *pattern* across all 50 dimensions
that encodes meaning.

In [ ]:
# Look at the raw vector for a word
vector = model["king"]
print(f"Vector for 'king': {vector[:10]}...  (showing first 10 of {len(vector)} dimensions)")
print(f"Type: {type(vector)}, shape: {vector.shape}")

## Part 2: Word Similarity

Words that appear in similar contexts end up with similar vectors.
We can use `most_similar()` to find the words whose vectors are closest
to a given word (measured by cosine similarity).

In [ ]:
# What words are most similar to "king"?
model.most_similar("king", topn=10)

In [ ]:
# Try some linguistics-related words
for word in ["language", "phoneme", "syntax", "vowel", "dialect"]:
    neighbors = model.most_similar(word, topn=5)
    neighbor_words = [w for w, _ in neighbors]
    print(f"{word:12s} → {', '.join(neighbor_words)}")

In [ ]:
# We can also measure the similarity between two specific words
pairs = [
    ("cat", "dog"),
    ("cat", "car"),
    ("good", "great"),
    ("good", "bad"),
    ("king", "queen"),
    ("king", "banana"),
]

for word1, word2 in pairs:
    sim = model.similarity(word1, word2)
    print(f"{word1:8s} — {word2:8s}  similarity: {sim:.3f}")

### Discussion

- "good" and "bad" have relatively high similarity. Why might that be?
  (Hint: think about the distributional hypothesis — what contexts do
  "good" and "bad" appear in?)
- Are there any results that surprise you?

## Part 3: Analogies

One of the most famous properties of word vectors: you can do arithmetic
with them and get meaningful results.

The classic example: **king − man + woman ≈ queen**

In vector space, this means: "take the concept of 'king', subtract the
'man-ness', add 'woman-ness', and you get 'queen'."

In gensim, we express this with `positive` (vectors to add) and
`negative` (vectors to subtract).

In [ ]:
# king - man + woman = ?
model.most_similar(positive=["king", "woman"], negative=["man"], topn=5)

In [ ]:
# Try some more analogies
analogies = [
    # (a, b, c) → a is to b as c is to ___?
    ("paris", "france", "tokyo"),
    ("bigger", "big", "slower"),
    ("walking", "walked", "swimming"),
]

for a, b, c in analogies:
    # a - b + c = ?
    # In other words: a is to b as ___ is to c
    results = model.most_similar(positive=[a, c], negative=[b], topn=3)
    result_words = [w for w, _ in results]
    print(f"{b} is to {a} as {c} is to ___?  → {', '.join(result_words)}")

### Try your own!

Use the cell below to try your own analogies or similarity queries.

In [ ]:
# Try your own queries here!
# model.most_similar("your_word")
# model.most_similar(positive=["word1", "word2"], negative=["word3"])

## Part 4: Visualizing Word Vectors with PCA

Each word vector has 50 dimensions — we can't visualize that directly.
But we can use **PCA** (Principal Component Analysis) to reduce the
vectors down to 2 dimensions while preserving as much of the structure
as possible.

This lets us plot words on a 2D graph and see whether semantically
related words cluster together.

In [ ]:
# Choose groups of semantically related words
word_groups = {
    "animals": ["cat", "dog", "horse", "fish", "bird", "mouse", "lion"],
    "colors": ["red", "blue", "green", "yellow", "black", "white", "purple"],
    "countries": ["france", "germany", "japan", "china", "brazil", "canada", "india"],
    "food": ["bread", "rice", "cheese", "chicken", "apple", "soup", "cake"],
}

# Collect all words and their vectors
all_words = []
all_vectors = []
group_labels = []

for group_name, words in word_groups.items():
    for word in words:
        if word in model:
            all_words.append(word)
            all_vectors.append(model[word])
            group_labels.append(group_name)

all_vectors = np.array(all_vectors)
print(f"Collected {len(all_words)} words across {len(word_groups)} groups")

In [ ]:
# Reduce to 2D with PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(all_vectors)

# Plot
plt.figure(figsize=(12, 8))

colors = {"animals": "tab:blue", "colors": "tab:red", "countries": "tab:green", "food": "tab:orange"}

for i, word in enumerate(all_words):
    group = group_labels[i]
    plt.scatter(coords[i, 0], coords[i, 1], color=colors[group], s=60)
    plt.annotate(
        word,
        (coords[i, 0], coords[i, 1]),
        fontsize=10,
        ha="center",
        va="bottom",
        color=colors[group],
    )

# Add a legend
for group_name, color in colors.items():
    plt.scatter([], [], color=color, label=group_name, s=60)
plt.legend(fontsize=11)

plt.title("Word Vectors Projected to 2D with PCA")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.tight_layout()
plt.show()

### Discussion

- Do the groups cluster together as you'd expect?
- Some words might appear "between" groups — why might that be?
- Remember: we're projecting from 50 dimensions to 2. A lot of structure
  is lost! Words that look close in 2D might not be as close in the
  original 50D space, and vice versa.

### Challenge: Make your own plot

<details>
<summary>Challenge: Linguistic word groups</summary>

Try making a PCA plot with linguistically interesting word groups. For
example:

```python
word_groups = {
    "body_parts": ["hand", "foot", "eye", "ear", "nose", "mouth", "head"],
    "kinship": ["mother", "father", "sister", "brother", "daughter", "son", "aunt"],
    "emotions": ["happy", "sad", "angry", "afraid", "surprised", "disgusted", "proud"],
    "languages": ["english", "french", "spanish", "chinese", "arabic", "japanese", "german"],
}
```

Do these cluster the way you'd expect? Are there any interesting outliers?
</details>

## Summary

| Concept | Key idea | Tool |
|---------|----------|------|
| Word vectors | Words as lists of numbers; similar words → similar vectors | `gensim` |
| Word2Vec | Learns vectors from context ("you know a word by its neighbors") | Pre-trained models |
| Analogies | Vector arithmetic captures relationships (king − man + woman ≈ queen) | `most_similar()` |
| PCA | Reduce high-dimensional vectors to 2D for visualization | `sklearn.decomposition.PCA` |
| BERT | Contextual vectors — same word gets different vectors in different sentences | `transformers` (HuggingFace) |

**For your projects**: If your project involves comparing words or
exploring semantic relationships, pre-trained word vectors are a powerful
and accessible tool. You can measure semantic similarity, find related
words, or visualize how words in your domain relate to each other.